# Smart Grocery & Meal Predictor

Given a **budget**, a **daily calorie goal**, a **daily protein goal**, a **dietary preference**,
and your **household size**, this notebook:

1. Picks a set of groceries (a "meal plan") that hits your nutrition targets at low cost, respecting your diet and exclusions
2. Forecasts grocery prices multiple weeks ahead, item by item
3. Tells you which items to buy now vs. wait on, based on the price trend
4. Forecasts your total weekly grocery spend
5. Exports a shopping list you can actually take to the store

Only **pandas, numpy, scikit-learn, and matplotlib** are used — no other libraries.

The sample data here (`food_dataset.csv`, `grocery_prices.csv`) is realistic but synthetic, so the notebook runs standalone.
Swap in real data when you're ready — see the **"Using real data"** note at the bottom for Kaggle/USDA sources.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load & clean the data

In [ ]:
food_data = pd.read_csv("food_dataset.csv")     # per-item price + nutrition + Veg/Non-Veg (for meal planning)
grocery_data = pd.read_csv("grocery_prices.csv") # weekly prices per item (for forecasting)

print("Food dataset:", food_data.shape)
print("Grocery price dataset:", grocery_data.shape)
food_data.head()

In [ ]:
print("Duplicate rows:", food_data.duplicated().sum())
print("\nMissing values:\n", food_data.isnull().sum())

food_data = food_data.drop_duplicates().reset_index(drop=True)
print("\nCleaned shape:", food_data.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(food_data["Food"], food_data["Price"], color="#4C72B0")
axes[0].set_title("Food Prices")
axes[0].set_ylabel("Price (Rs)")
axes[0].tick_params(axis="x", rotation=75)

axes[1].bar(food_data["Food"], food_data["Calories"], color="#DD8452")
axes[1].set_title("Calories per Item")
axes[1].set_ylabel("Calories")
axes[1].tick_params(axis="x", rotation=75)

plt.tight_layout()
plt.show()

## 2. Personalize your plan

This is the interactive part — your answers here actually change which foods the planner is allowed to choose from,
and how big the targets are.

In [ ]:
# Basic targets
budget = float(input("Weekly grocery budget (Rs): "))
calorie_goal = int(input("Daily calorie goal per person (kcal): "))
protein_goal = int(input("Daily protein goal per person (g): "))

# Personalization
diet_pref = input("Dietary preference — any / veg / non-veg: ").strip().lower()
exclude_raw = input("Any foods to avoid (allergies/dislikes), comma-separated, or leave blank: ")
household_size = int(input("How many people is this for? ") or 1)

exclude_list = [x.strip().lower() for x in exclude_raw.split(",") if x.strip()]

# Filter the food catalog to what's actually allowed
available = food_data.copy()
if diet_pref == "veg":
    available = available[available["Type"] == "Veg"]
if exclude_list:
    pattern = "|".join(exclude_list)
    available = available[~available["Food"].str.lower().str.contains(pattern)]

# Scale nutrition targets for the whole household
household_calorie_goal = calorie_goal * household_size
household_protein_goal = protein_goal * household_size

print(f"\nFoods available after filtering: {len(available)} / {len(food_data)}")
print(f"Household targets: {household_calorie_goal} kcal, {household_protein_goal} g protein, budget Rs {budget:.2f}")

if available.empty:
    print("\nWarning: no foods left after filtering — loosen your exclusions or diet preference.")

## 3. Meal planner (greedy optimization)

Two-step greedy heuristic, built with just pandas/numpy:

1. **Diversity pass** — take the cheapest *available* item from each food category, so the plan spans grains, dairy, protein, vegetables, and fruit.
2. **Efficiency pass** — rank the rest by "nutrition delivered per rupee" toward the household's calorie + protein targets, and keep adding the most efficient ones until both goals are met or the budget runs out.

Because this only picks from `available` (already filtered by diet/exclusions), the plan will always respect what you told it in Section 2.

In [ ]:
picked_idx = []
for cat in available["Category"].unique():
    cat_rows = available[available["Category"] == cat]
    picked_idx.append(cat_rows["Price"].idxmin())
plan_idx = list(dict.fromkeys(picked_idx))  # de-dupe, keep order

spent = available.loc[plan_idx, "Price"].sum()
got_calories = available.loc[plan_idx, "Calories"].sum()
got_protein = available.loc[plan_idx, "Protein"].sum()

if spent > budget:
    print(f"Warning: just one item from each food group already costs Rs {spent:.2f}, "
          f"which is over your Rs {budget:.2f} budget.")
    print("Showing the cheapest possible diverse plan anyway — raise the budget to fix this.")
else:
    remaining = available.drop(index=plan_idx).copy()
    remaining["Efficiency"] = (
        remaining["Calories"] / max(household_calorie_goal, 1) + remaining["Protein"] / max(household_protein_goal, 1)
    ) / remaining["Price"]
    remaining = remaining.sort_values("Efficiency", ascending=False)

    for idx, row in remaining.iterrows():
        if got_calories >= household_calorie_goal and got_protein >= household_protein_goal:
            break
        if spent + row["Price"] > budget:
            continue
        plan_idx.append(idx)
        spent += row["Price"]
        got_calories += row["Calories"]
        got_protein += row["Protein"]

plan = available.loc[plan_idx].copy()
plan[["Food", "Category", "Type", "Price", "Calories", "Protein"]]

In [ ]:
meal_cost = plan["Price"].sum()
meal_calories = plan["Calories"].sum()
meal_protein = plan["Protein"].sum()
remaining_budget = budget - meal_cost

print(f"Total cost      : Rs {meal_cost:.2f}")
print(f"Calories        : {meal_calories:.0f}  (household goal: {household_calorie_goal})")
print(f"Protein         : {meal_protein:.1f} g (household goal: {household_protein_goal} g)")
print(f"Remaining budget: Rs {remaining_budget:.2f}")
print()
print("Calorie goal met:", "Yes" if meal_calories >= household_calorie_goal else "No — try raising the budget")
print("Protein goal met:", "Yes" if meal_protein >= household_protein_goal else "No — try raising the budget")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie([meal_calories, meal_protein * 4], labels=["Calories (kcal)", "Protein (kcal-equiv)"],
            autopct="%1.1f%%", colors=["#4C72B0", "#55A868"])
axes[0].set_title("Nutrition Split of the Plan")

axes[1].bar(plan["Food"], plan["Price"], color="#55A868")
axes[1].set_title("Cost of Recommended Items")
axes[1].set_ylabel("Price (Rs)")
axes[1].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

## 4. Price forecasting, multiple weeks ahead

Choose how far ahead to look. For each tracked item, two models — `LinearRegression` and `RandomForestRegressor` —
are trained on weekly price history and evaluated on a held-out test split; whichever generalizes better (lower MAE)
is used for the actual forecast.

In [ ]:
weeks_ahead = int(input("How many weeks ahead would you like to forecast? (1-8): ") or 1)
weeks_ahead = max(1, min(weeks_ahead, 8))

item_cols = [c for c in grocery_data.columns if c not in ("Week", "Date", "DataType")]
X = grocery_data[["Week"]]
last_week = grocery_data["Week"].max()
future_weeks = pd.DataFrame({"Week": np.arange(last_week + 1, last_week + 1 + weeks_ahead)})

price_models = {}
model_names = {}
future_prices = {"Week": future_weeks["Week"].values}

for item in item_cols:
    y = grocery_data[item]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    lr = LinearRegression().fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)
    lr_mae = mean_absolute_error(y_test, lr.predict(X_test))
    rf_mae = mean_absolute_error(y_test, rf.predict(X_test))

    best_model, best_name = (lr, "LinearRegression") if lr_mae <= rf_mae else (rf, "RandomForest")
    best_model.fit(X, y)  # refit on all available weeks for the final forecast

    price_models[item] = best_model
    model_names[item] = best_name
    future_prices[item] = best_model.predict(future_weeks)

future_df = pd.DataFrame(future_prices)
print(f"Forecast for the next {weeks_ahead} week(s):")
future_df

In [ ]:
# Chart only the items actually in your meal plan, not the whole 46-item catalog -
# with a large catalog like this one, an all-items grid becomes too tall to be usable.
plan_names_for_chart = plan["Food"].str.split(" ").str[0]
tracked_plan_items = [name for name in plan_names_for_chart if name in item_cols]

if tracked_plan_items:
    n_items = len(tracked_plan_items)
    n_cols = 3
    n_rows = -(-n_items // n_cols)  # ceiling division
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten() if n_items > 1 else [axes]

    for ax, item in zip(axes, tracked_plan_items):
        ax.plot(grocery_data["Week"], grocery_data[item], marker="o", label="Actual")
        ax.plot(future_df["Week"], future_df[item], marker="o", linestyle="--", color="red", label="Forecast")
        ax.set_title(f"{item} ({model_names[item]})")
        ax.set_xlabel("Week")
        ax.set_ylabel("Price (Rs)")
        ax.legend(fontsize=8)

    for ax in axes[n_items:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("None of this week's recommended items have price history to chart.")

print(f"\n({len(item_cols)} items tracked in total across the catalog — see the 'Buy now or wait?' "
      f"table below for all of them, not just the {len(tracked_plan_items)} in your plan.)")

In [ ]:
plan_names = plan["Food"].str.split(" ").str[0]
tracked_plan_items = [name for name in plan_names if name in item_cols]
untracked_plan_cost = plan[~plan_names.isin(item_cols)]["Price"].sum()

if tracked_plan_items:
    plan_hist_cost = grocery_data[tracked_plan_items].sum(axis=1) + untracked_plan_cost
    plan_future_cost = future_df[tracked_plan_items].sum(axis=1) + untracked_plan_cost
else:
    plan_hist_cost = pd.Series(untracked_plan_cost, index=grocery_data.index)
    plan_future_cost = pd.Series(untracked_plan_cost, index=future_df.index)

plt.plot(grocery_data["Week"], plan_hist_cost, marker="o", label="Actual (this plan's tracked items)")
plt.plot(future_df["Week"], plan_future_cost, marker="o", linestyle="--", color="red", label="Forecast")
plt.xlabel("Week")
plt.ylabel("Meal Plan Cost (Rs)")
plt.title("Your Meal Plan's Weekly Cost Trend & Forecast")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

predicted_plan_total = plan_future_cost.iloc[-1]
print(f"Predicted cost of THIS meal plan, week {int(future_df['Week'].iloc[-1])}: Rs {predicted_plan_total:.2f}")
if untracked_plan_cost > 0:
    print(f"(Rs {untracked_plan_cost:.2f} of that is items with no price history, held at their known price)")

## 5. Buy now or wait?

For every tracked item, compare today's price to the price forecast at the end of your chosen horizon.
A rise of more than 2% suggests buying now; a drop of more than 2% suggests it's fine to wait.

In [ ]:
rows = []
for item in item_cols:
    last_price = grocery_data[item].iloc[-1]
    future_price = future_df[item].iloc[-1]
    pct_change = (future_price - last_price) / last_price * 100
    if pct_change > 2:
        verdict = "Buy now"
    elif pct_change < -2:
        verdict = "Wait"
    else:
        verdict = "No urgency"
    rows.append({
        "Item": item,
        "Current price": round(last_price, 2),
        f"Price in {weeks_ahead}wk": round(future_price, 2),
        "Change %": round(pct_change, 1),
        "Suggestion": verdict,
    })

advice_df = pd.DataFrame(rows).sort_values("Change %", ascending=False).reset_index(drop=True)
advice_df

## 6. Tie it together

If any item in this week's recommended meal plan is also one of the items we track prices for,
show what it's projected to cost by the end of your chosen forecast horizon.

In [ ]:
plan_item_names = plan["Food"].str.split(" ").str[0]  # "Eggs (6 pcs)" -> "Eggs"
tracked_items = [name for name in plan_item_names if name in item_cols]

if tracked_items:
    tracked = advice_df[advice_df["Item"].isin(tracked_items)]
    display(tracked)
    projected_cost = tracked[f"Price in {weeks_ahead}wk"].sum()
    print(f"\nProjected cost for these tracked items in {weeks_ahead} week(s): Rs {projected_cost:.2f}")
else:
    print("None of this week's recommended items overlap with the price-tracking dataset.")

## 7. Recommendations & dashboard

In [ ]:
print("=" * 45)
print("Recommendations")
print("=" * 45)

if remaining_budget > 100:
    print(f"You have Rs {remaining_budget:.2f} left — consider adding more fruit or vegetables.")
else:
    print("Budget is nearly used up — the plan above is tight but on-target.")

if meal_protein < household_protein_goal:
    print("\nProtein is short of goal. Consider adding: Paneer, Eggs (if non-veg), Chickpeas, or Soybean Chunks.")
else:
    print("\nProtein goal met for this plan.")

top_riser = advice_df.iloc[0]
if top_riser["Change %"] > 2:
    print(f"\n{top_riser['Item']} is trending up the most (+{top_riser['Change %']}%) — buy it this week rather than waiting.")
else:
    print("\nNo item is trending sharply upward — no urgency to stock up on anything specific.")

In [ ]:
print("=" * 50)
print("SMART GROCERY & MEAL PREDICTOR — SUMMARY")
print("=" * 50)
print(f"Household size            : {household_size}")
print(f"Diet preference           : {diet_pref}")
print(f"Meal plan cost            : Rs {meal_cost:.2f}")
print(f"Remaining budget          : Rs {remaining_budget:.2f}")
print(f"Calories / Protein        : {meal_calories:.0f} kcal / {meal_protein:.1f} g")
print(f"Predicted meal plan cost (wk {int(future_df['Week'].iloc[-1])}) : Rs {predicted_plan_total:.2f}")
print(f"Item to watch             : {top_riser['Item']} ({top_riser['Suggestion']})")
print("=" * 50)

## 8. Export your shopping list

Saves the recommended plan and the price forecast to CSV files you can actually take with you or open in Excel.

In [ ]:
plan_export = plan[["Food", "Category", "Type", "Price", "Calories", "Protein"]].copy()
plan_export.to_csv("my_shopping_list.csv", index=False)

forecast_export = future_df.copy()
forecast_export.to_csv("my_price_forecast.csv", index=False)

print("Saved: my_shopping_list.csv")
print("Saved: my_price_forecast.csv")

## Using real data

To move past the synthetic CSVs here:

- **Nutrition data** → [USDA FoodData Central API](https://fdc.nal.usda.gov/api-guide.html) (free API key). Replace `food_dataset.csv` with items pulled from there — same columns (`Food, Category, Price, Calories, Protein, Type`), just add your own local prices since USDA doesn't have those.
- **Grocery price history** → search Kaggle for "grocery sales dataset" / "supermarket sales" for real historical prices, or the [USDA Food Price Outlook](https://www.ers.usda.gov/data-products/food-price-outlook) for category-level trend data. Reshape it to match `grocery_prices.csv` (one column per item, one row per week).
- Once you have more weeks of real data (52+ ideally), the forecasting models will have much more signal to work with than the 12 synthetic weeks here — right now `Week number` is the only feature; with real data you could add month, holidays, or rolling averages as extra features.